# Part I exercises — tensor network representations

Roughly 30 minutes. Each exercise states the point it is making, then asks you to make
a prediction *before* running anything. The prediction matters more than the answer:
if you are surprised, you have learnt something, and if you are not, you can move on
quickly.

`assert` cells at the end of each exercise tell you whether it worked. Solutions are at
the bottom — try not to look at them, but do look rather than get stuck.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import qtade_quimb as qq
import qtade_tn as tn

plt.rcParams.update({"figure.figsize": (8, 3), "axes.grid": True, "grid.alpha": 0.3})

---
## Exercise 1 — rank is not about resolution

**Point:** the quantics bond dimension measures how many scales are *coupled*, not how
many grid points there are.

For each function below, write down your guess for the bond dimension *before*
measuring. Then fill in `chi_of` and check.

In [ ]:
def chi_of(f, eps=1e-8):
    """Return the maximum QTT bond dimension of a sampled function."""
    # TODO: one line. Build the train and read off its ranks.
    raise NotImplementedError


n = 14
x = np.linspace(0, 1, 2 ** n, endpoint=False)

candidates = {
    "1": np.ones_like(x),
    "x": x,
    "x**2": x ** 2,
    "exp(5x)": np.exp(5 * x),
    "exp(5x) + exp(-3x)": np.exp(5 * x) + np.exp(-3 * x),
    "sin(2 pi x) * exp(-x)": np.sin(2 * np.pi * x) * np.exp(-x),
    "sin(2 pi x) + sin(64 pi x)": np.sin(2 * np.pi * x) + np.sin(64 * np.pi * x),
}
# for name, f in candidates.items():
#     print(f"{name:>28}: chi = {chi_of(f)}")

Two things to explain to your neighbour once you have the numbers:

1. Why is `exp(5x) + exp(-3x)` cheaper than you might fear?
2. `sin(2 pi x) + sin(64 pi x)` contains a very fine oscillation. Is it expensive?
   Why not?

---
## Exercise 2 — where does rank actually come from?

**Point:** the enemy is scale coupling, not sharpness.

Sweep the width of a Gaussian bump from wide to narrower than one grid cell, and plot
the bond dimension. Predict the shape of the curve first: monotone? peaked? flat?

In [ ]:
n = 12
x = np.linspace(0, 1, 2 ** n, endpoint=False)
widths = np.logspace(-4, -0.5, 25)

# TODO: for each width, build exp(-((x-0.5)/w)**2) as a QTT and record max chi.
# Then plot chi against w on a log x-axis.

You should find the curve is **not** monotone. Explain the two ends:
what is happening when the bump is wider than the domain, and what is happening when it
is narrower than a grid cell?

---
## Exercise 3 — build the periodic shift

**Point:** boundary conditions live in the boundary vectors of the MPO, nowhere else.

`qtt_shift` builds the non-periodic shift by forcing the left carry bond to state 0 —
"no overflow off the top". Allow that overflow and you get the periodic shift, i.e. the
cyclic permutation matrix.

Copy the body of `qtt_shift` and change exactly one line.

In [ ]:
def qtt_shift_periodic(n):
    """Cyclic shift: S[i, j] = 1 iff i == (j + 1) mod 2**n."""
    core = np.zeros((2, 2, 2, 2))            # [carry_out, i, j, carry_in]
    for c_in in (0, 1):
        for j in (0, 1):
            i = (j + c_in) % 2
            c_out = 1 if (j == 1 and c_in == 1) else 0
            core[c_out, i, j, c_in] = 1.0
    cores = [core.copy() for _ in range(n)]
    # TODO: close the LEFT edge so that an overflowing carry wraps around instead of
    # being discarded, and close the right edge as before.
    raise NotImplementedError


# n = 4
# P = qtt_shift_periodic(n)
# ref = np.roll(np.eye(2 ** n), 1, axis=0)
# assert np.allclose(tn.mpo_full(P), ref), "not the cyclic shift yet"
# print("periodic shift ranks:", tn.mpo_ranks(P))

---
## Exercise 4 — the cost of forgetting to round

**Point:** rounding is the algorithm, not an optimisation.

Run explicit Euler for the heat equation *without* the rounding step and find how many
timesteps you get before the cores exceed a memory budget of 256 MiB. Compare with the
prediction $\chi_k = 3^k \chi_0$.

In [ ]:
n = 10
h = 2.0 ** -n
xg = np.linspace(0, 1, 2 ** n, endpoint=False)
L = tn.qtt_laplacian(n, dx=h)
step = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, 0.4 * h ** 2)), 1e-13)
u0 = tn.qtt_from_vector(np.sin(2 * np.pi * xg), eps=1e-12)

# TODO: loop applying `step` without rounding. Check the PREDICTED size of the next
# state (ranks multiply by 3) before you compute it -- otherwise the cell allocates the
# thing you were trying to avoid and takes the kernel with it.
# Print k, max(tn.tt_ranks(u)), the memory, and 3**k * chi_0 at each step.

Now repeat *with* `tn.tt_round(..., eps=1e-8)` after each application and confirm the
bond dimension stays flat while the answer stays the same to eight digits.

---
## Exercise 5 — bit ordering in two dimensions

**Point:** stacked versus interleaved is a modelling decision, and which one wins
depends on the field. There is no universally better ordering, and anyone who tells you
otherwise has only tried one kind of field.

`qq.from_grid` takes `ordering="interleaved"` (x1 y1 x2 y2 ...) or `"stacked"`
(x1 ... xn y1 ... yn). Compare the two on:

* a **separable** field, $e^{-40(x-\frac12)^2}\,e^{-40(y-\frac12)^2}$
* a **curved sharp** field, the indicator of a disc
* a **multiscale** field, synthetic turbulence with a $k^{-5/3}$ spectrum

Predict the winner for each. Then compare them twice: at matched *tolerance*
(`eps=1e-8`, let $\chi$ be whatever it needs) and at matched *bond dimension*
(`chi_max=32`, measure the error). The two comparisons do not agree, and understanding
why is the exercise.

In [ ]:
m = 8
N = 2 ** m
xx = np.linspace(0, 1, N, endpoint=False)
X, Y = np.meshgrid(xx, xx, indexing="ij")
R = np.hypot(X - 0.5, Y - 0.5)

rng = np.random.default_rng(3)
kf = np.fft.fftfreq(N) * N
KX, KY = np.meshgrid(kf, kf, indexing="ij")
Kmag = np.hypot(KX, KY)
Kmag[0, 0] = 1.0
spec = Kmag ** (-5 / 6)
spec[Kmag > N / 3] = 0.0
phase = rng.standard_normal((N, N)) + 1j * rng.standard_normal((N, N))
turbulent = np.real(np.fft.ifft2(spec * phase))
turbulent /= turbulent.std()

fields = {
    "separable gaussian": np.exp(-40 * (X - 0.5) ** 2) * np.exp(-40 * (Y - 0.5) ** 2),
    "disc indicator": (R < 0.25).astype(float),
    "k^-5/3 turbulence": turbulent,
}
# TODO (a): for each field print max chi under both orderings at eps=1e-8.
# TODO (b): for each field compress with chi_max=32 under both orderings and print the
#           relative L2 error. Use qq.to_grid for interleaved and
#           tn.tt_full(...).reshape(N, N) for stacked.

Once you have both tables, answer these:

1. Why does stacking win so decisively on the separable field? (Hint: what is the rank
   across the single $x|y$ cut for a product $f(x)g(y)$?)
2. At matched tolerance the multiscale field looks cheaper stacked. At matched bond
   dimension it is more accurate interleaved. Both are true — what is each measuring?
3. Which comparison is the one you care about when you have a fixed memory budget?

This is the same question Pisoni *et al.* (2026) ask in 3D, where the diagnostic that
separates the orderings is not $L_2$ error at all but the probability distribution of
velocity increments. The spectrum looks fine long after the flow does not.

---
## Solutions

Resist for at least one honest attempt per exercise.

In [ ]:
# --- Exercise 1 ---
def chi_of_solution(f, eps=1e-8):
    return max(tn.tt_ranks(tn.qtt_from_vector(f, eps=eps)))


for name, f in candidates.items():
    print(f"{name:>28}: chi = {chi_of_solution(f)}")
print("\nexp(5x)+exp(-3x) is rank 2: a sum of m exponentials has rank at most m.")
print("sin(2 pi x) + sin(64 pi x) is still cheap: both frequencies are resolved,")
print("and each sine is rank 2 on its own, so the sum is at most 4.")

In [ ]:
# --- Exercise 2 ---
n2 = 12
x2 = np.linspace(0, 1, 2 ** n2, endpoint=False)
ws = np.logspace(-4, -0.5, 25)
chis = [chi_of_solution(np.exp(-((x2 - 0.5) / w) ** 2)) for w in ws]
plt.semilogx(ws, chis, "o-")
plt.axvline(2.0 ** -n2, ls="--", c="k", lw=1)
plt.text(2.0 ** -n2 * 1.2, max(chis) * 0.9, "one grid cell", fontsize=9)
plt.xlabel("Gaussian width"), plt.ylabel("max bond dimension")
plt.title("rank peaks where the feature spans several, but not all, scales")
plt.tight_layout()
print("Wide bump: smooth, few scales involved, cheap.")
print("Sub-cell bump: a single spike, which is a product of delta functions per digit,")
print("so it is cheap again. The expensive regime is in between.")

In [ ]:
# --- Exercise 3 ---
def qtt_shift_periodic_solution(n):
    core = np.zeros((2, 2, 2, 2))
    for c_in in (0, 1):
        for j in (0, 1):
            i = (j + c_in) % 2
            c_out = 1 if (j == 1 and c_in == 1) else 0
            core[c_out, i, j, c_in] = 1.0
    cores = [core.copy() for _ in range(n)]
    # periodic: the outgoing carry wraps back in, so trace the bond instead of
    # forcing it to zero. With the right edge fixed to carry-in = 1, keeping BOTH
    # left states and summing them is exactly that wrap-around.
    cores[0] = cores[0].sum(axis=0, keepdims=True)
    cores[-1] = cores[-1][:, :, :, 1:2]
    return cores


P = qtt_shift_periodic_solution(4)
assert np.allclose(tn.mpo_full(P), np.roll(np.eye(16), 1, axis=0))
print("periodic shift ranks:", tn.mpo_ranks(P), "- same rank 2, one different edge")

In [ ]:
# --- Exercise 4 ---
u = [c.copy() for c in u0]
chi0 = max(tn.tt_ranks(u))
budget = 256 * 2 ** 20
k = 0
while 9 * tn.tt_size(u) * 8 < budget:          # next step multiplies memory by ~9
    u = tn.mpo_apply(step, u)
    k += 1
    print(f"k={k:>2}  chi={max(tn.tt_ranks(u)):>6,d}  predicted 3^k*chi0="
          f"{3 ** k * chi0:>6,d}  memory={tn.tt_size(u) * 8 / 2 ** 20:>8.2f} MiB")
print(f"\n{k} steps before the 256 MiB budget. With rounding:")
u = [c.copy() for c in u0]
for _ in range(k + 20):
    u = tn.tt_round(tn.mpo_apply(step, u), eps=1e-8)
print(f"  after {k + 20} steps: chi = {max(tn.tt_ranks(u))}, "
      f"{tn.tt_size(u) * 8 / 2 ** 10:.1f} KiB")

In [ ]:
# --- Exercise 5 ---
print("(a) matched tolerance, eps = 1e-8")
for name, f in fields.items():
    ci = qq.from_grid(f, eps=1e-8, ordering="interleaved")
    cs = qq.from_grid(f, eps=1e-8, ordering="stacked")
    print(f"  {name:>20}:  interleaved chi={max(tn.tt_ranks(ci)):>4} "
          f"({tn.tt_size(ci):>7,d} params)   "
          f"stacked chi={max(tn.tt_ranks(cs)):>4} ({tn.tt_size(cs):>7,d} params)")

print("\n(b) matched bond dimension, chi_max = 32")
for name, f in fields.items():
    ci = qq.from_grid(f, eps=1e-14, chi_max=32, ordering="interleaved")
    cs = qq.from_grid(f, eps=1e-14, chi_max=32, ordering="stacked")
    ei = np.linalg.norm(qq.to_grid(ci) - f) / np.linalg.norm(f)
    es = np.linalg.norm(tn.tt_full(cs).reshape(N, N) - f) / np.linalg.norm(f)
    print(f"  {name:>20}:  interleaved L2={ei:.4f}   stacked L2={es:.4f}")

print("""
1. A product f(x)g(y) has rank exactly 1 across the stacked x|y cut, by definition of
   a product. Stacking is therefore the optimal ordering for anything separable, and
   many textbook test functions are separable -- which is how the folklore that
   "stacked is fine" survives.
2. Matched tolerance measures how many numbers each ordering needs to be exact.
   Matched bond dimension measures how good each ordering is when you have already
   decided what you can afford. For a multiscale field these disagree: stacking is
   more efficient asymptotically, interleaving degrades more gracefully.
3. The second. You never get to choose the tolerance in a large simulation; you choose
   chi, because chi is what fits in memory.""")